# 02 Data Wrangling and Visualization — Exercises

Complete the exercises below using the Pine and Cypress Nursing Home Legionnaires' disease line list (`legionella_outbreak.csv`).

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (avoid Chinese labels showing as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# Plotly: make sure interactive charts still render during a static build (jupyter-book build)
pio.renderers.default = "notebook"


## Question 1: Read in and inspect the data

1. Read in `data/synthetic/legionella_outbreak.csv`
2. Print the first 5 rows and all column names
3. Answer: How many records? How many columns? Which columns have missing values?

In [ ]:
# TODO: read the CSV
# TODO: print df.head() and df.columns.tolist()
# TODO: use df.info() to see missing values

## Question 2: Date conversion and derived variables

1. Convert `symptom_onset_date` and `hospitalization_date` to datetime
2. Create an `infected` column (`clinical_severity != 'not_ill'` → 1, otherwise 0)
3. Compute `onset_to_hosp_days` (hospitalization date − onset date)
4. Print the `case_id`, `symptom_onset_date`, and `onset_to_hosp_days` of the first 10 infected people

In [ ]:
# TODO: date conversion
# TODO: create the infected column
# TODO: compute onset_to_hosp_days
# TODO: filter the infected and print the specified columns

## Question 3: Epidemic curve

Draw an epidemic curve with matplotlib:
- Take only the infected
- X-axis: `symptom_onset_date`
- Y-axis: number of new cases per day
- Add a chart title and axis labels

In [ ]:
# TODO: filter the infected
# TODO: groupby symptom_onset_date and count
# TODO: draw a bar chart
# TODO: add a title and axis labels

## Question 4: Attack rate by wing comparison chart

1. Use `groupby(["floor", "wing"])` to compute the number of residents and number infected in each wing
2. Compute the attack rate (%)
3. Draw a sorted bar chart with seaborn, labeling the number above each bar

In [ ]:
# TODO: groupby + agg
# TODO: compute the attack rate
# TODO: sort + draw a barplot
# TODO: label the numbers

## Question 5 (Challenge): Interactive stratified epidemic curve

Draw a stratified epidemic curve with Plotly:
- Color by `floor`
- Use a stacked bar chart (`barmode="stack"`)
- Add a title and axis labels

Observe: are the epidemic peaks of the three floors synchronized? What is the epidemiological significance of this?

In [ ]:
# TODO: groupby symptom_onset_date + floor and count
# TODO: draw an interactive chart with px.bar

## Question 6: Frequency table and pivot table

1. Use `value_counts()` to list the frequency distribution of `clinical_severity` (with percentages)
2. Use `pd.pivot_table()` to build an attack-rate table of **wing × floor** (hint: `values="infected"`, `aggfunc="mean"`)
3. Add `margins=True` to show subtotals

In [ ]:
# TODO: value_counts() to list the severity distribution
# TODO: value_counts(normalize=True) to add percentages
# TODO: pd.pivot_table() to build the wing × floor attack-rate table
# TODO: add margins=True

## Question 7: Method Chaining

Use method chaining (a single-line pipeline) to complete the following analysis:
1. Filter infected people aged 70+ (hint: `.query("infected == 1 and age >= 70")`)
2. Group by `floor`
3. Use `.agg()` to compute both the case count and death count at once
4. Add a case-fatality-rate column (`.assign(cfr=lambda d: ...)`)
5. Sort by case fatality rate from high to low

In [ ]:
# TODO: use method chaining to complete the analysis above
# Hint: (df.query(...).groupby(...).agg(...).assign(...).sort_values(...))

## Question 8: Joining data and text cleaning

1. Simulate a lab-results DataFrame (`lab`) containing `case_id` and `ct_value`
2. Use `pd.merge()` to left-join `lab` onto `df`
3. Use `.str.strip().str.upper()` to standardize the case of the `wing` column
4. Use `drop_duplicates("case_id")` to remove duplicate reports
5. Use `nlargest(5, "age")` to find the 5 oldest infected people

In [ ]:
# TODO: build the lab DataFrame (case_id, ct_value)
# TODO: pd.merge(df, lab, on="case_id", how="left")
# TODO: str.strip().str.upper() to standardize wing
# TODO: drop_duplicates("case_id")
# TODO: nlargest(5, "age")

## Question 9: COVID-19 daily case-report line list wrangling and grouping (COVID-19 scenario)

A community screening site compiled a daily COVID-19 case-report line list (the code below generates `covid_df`), with columns for report date, vaccination status, clinical severity, and hospitalization status. Complete the following tasks:

1. Filter for cases with `test_result == "positive"` and save as `covid_cases`
2. Use `groupby("vaccination_status")` with `.agg()` to compute the number of cases and number hospitalized for each vaccination status
3. Add a `hospitalization_rate_pct` column (number hospitalized / number of cases × 100), and sort from high to low
4. Use `value_counts()` to list the frequency and percentage distribution of `severity` (clinical severity)
5. Compute the daily number of new positive cases by `report_date`, and use matplotlib to draw an epidemic curve (remember to add a title and axis labels)
6. Interpret: Is the hospitalization rate among the unvaccinated markedly higher than among the fully vaccinated? What does this mean for vaccination policy?

In [ ]:
import numpy as np

rng = np.random.default_rng(202)

n = 500
report_dates = pd.date_range("2026-03-01", periods=21, freq="D")

vaccination_status = rng.choice(
    ["未接種", "部分接種", "完整接種"], size=n, p=[0.30, 0.25, 0.45]
)
age = rng.integers(1, 90, size=n)
sex = rng.choice(["M", "F"], size=n)
report_date = report_dates[rng.integers(0, len(report_dates), size=n)]
test_result = rng.choice(["positive", "negative"], size=n, p=[0.55, 0.45])

# Set the clinical severity distribution by vaccination status (the unvaccinated have a higher severe-case rate)
sev_categories = ["none", "mild", "moderate", "severe"]
sev_probs = {
    "未接種": [0.35, 0.35, 0.20, 0.10],
    "部分接種": [0.55, 0.30, 0.11, 0.04],
    "完整接種": [0.70, 0.25, 0.04, 0.01],
}
severity = np.array(
    [rng.choice(sev_categories, p=sev_probs[v]) for v in vaccination_status]
)

hospitalized = np.isin(severity, ["moderate", "severe"]).astype(int)
extra_hosp = (severity == "mild") & (rng.random(n) < 0.05)
hospitalized = np.where(extra_hosp, 1, hospitalized)

covid_df = pd.DataFrame({
    "report_id": [f"C{i:04d}" for i in range(1, n + 1)],
    "report_date": report_date,
    "age": age,
    "sex": sex,
    "vaccination_status": vaccination_status,
    "test_result": test_result,
    "severity": severity,
    "hospitalized": hospitalized,
})

# TODO: filter for cases with test_result == "positive" and save as covid_cases
# TODO: use groupby("vaccination_status") + .agg() to compute the number of cases and number hospitalized per vaccination status
# TODO: add a hospitalization_rate_pct column (number hospitalized / number of cases * 100), and sort from high to low
# TODO: use value_counts() to list the frequency and percentage distribution of severity
# TODO: compute the daily number of new positive cases by report_date, and use matplotlib to draw an epidemic curve (with title and axis labels)
# TODO: print the conclusion: is the hospitalization rate markedly higher among the unvaccinated? What does this mean for vaccination policy?

## Question 10: Dengue Fever Surveillance Statistics by Region (dengue scenario, Challenge)

The health authority compiled an 8-week dengue fever case-report line list for five administrative districts in southern Taiwan (the code below generates `dengue_df`), along with each district's population (`region_population`). Complete the following tasks:

1. Use `groupby("region")` with `.agg()` to compute the number of cases and mean age for each district
2. Convert `region_population` into a DataFrame, and use `pd.merge()` to join it with the case statistics to compute the **incidence rate per 100,000 population** for each district (`n_cases / population * 100000`)
3. Add a derived `epi_week` column (hint: `dengue_df["symptom_onset_date"].dt.isocalendar().week`)
4. Use `groupby(["region", "epi_week"])` to tabulate the weekly case count for each district, and use `pivot_table()` or `.unstack()` to arrange it into a "region × week" table
5. (Challenge) Use seaborn to draw a bar chart of districts sorted by incidence rate, and label which week had the most cases in each district
6. Interpret: Is the district with the highest incidence rate per 100,000 population also the district with the most cases? Why might these not be the same?

In [ ]:
import numpy as np

rng = np.random.default_rng(818)

region_population = {
    "高雄市三民區": 34000,
    "高雄市苓雅區": 28000,
    "台南市北區": 21000,
    "台南市安南區": 19000,
    "屏東市": 15000,
}
regions = list(region_population.keys())

n_cases = 360
# Case-burden weight for each district (Sanmin and Annan have more standing-water containers and thus higher risk; weights are not proportional to population)
region_weights = [0.34, 0.10, 0.14, 0.32, 0.10]

case_region = rng.choice(regions, size=n_cases, p=region_weights)
onset_date = pd.Timestamp("2026-06-01") + pd.to_timedelta(
    rng.integers(0, 56, size=n_cases), unit="D"
)  # 8-week surveillance period
age = rng.integers(1, 95, size=n_cases)
sex = rng.choice(["M", "F"], size=n_cases)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(1, n_cases + 1)],
    "region": case_region,
    "symptom_onset_date": onset_date,
    "age": age,
    "sex": sex,
})

# TODO: use groupby("region") + .agg() to compute the number of cases and mean age for each district
# TODO: convert region_population into a DataFrame, and use pd.merge() to join it with the case statistics
# TODO: compute the incidence rate per 100,000 population: n_cases / population * 100000
# TODO: add a derived epi_week column (dengue_df["symptom_onset_date"].dt.isocalendar().week)
# TODO: use groupby(["region", "epi_week"]) to tabulate weekly case counts per district, and use pivot_table() or unstack() to arrange it into a table
# TODO: (Challenge) use seaborn to draw a bar chart of districts sorted by incidence rate, and label which week had the most cases in each district
# TODO: print the conclusion: is the district with the highest incidence rate also the district with the most cases? Why might these not be the same?